# Feature Extraction Pipeline — Test Notebook
Tests each stage of `semantic_structure/extractor.py`:
1. HTML parsing → `(token, tag)` pairs
2. Vocab building + GloVe loading
3. Ω' matrix construction

In [2]:
import sys
sys.path.insert(0, '..')

import numpy as np
import config
from semantic_structure.db import load_records
from semantic_structure.extractor import (
    _parse_html,
    _tokenize,
    build_corpus_vocab,
    extract_page,
)

print('Imports OK')
print(f'DB path:    {config.DB_PATH}')
print(f'GloVe path: {config.GLOVE_PATH}')
print(f'M={config.M}, N={config.N}')

Imports OK
DB path:    /Users/umair/Library/Application Support/mlops/pages.db
GloVe path: /Users/umair/mlops/models/semantic_structure/GloVe/dolma_300_2024_1.2M.100_combined.txt
M=512, K=300, N=16


## 1. Tokenizer

In [ ]:
cases = [
    ("Hello, World!",               ["hello", "world"]),
    ("PyTorch 2.0 docs",            ["pytorch", "2.0", "docs"]),
    ("don't stop",                  ["don't", "stop"]),
    ("multi-head attention",        ["multi-head", "attention"]),
    ("   extra   whitespace   ",    ["extra", "whitespace"]),
    ("",                            []),
]

all_passed = True
for text, expected in cases:
    result = _tokenize(text)
    status = "PASS" if result == expected else "FAIL"
    if status == "FAIL":
        all_passed = False
    print(f"{status}  input={repr(text)!r:40s}  got={result}")

print()
print("All tokenizer tests passed" if all_passed else "SOME TESTS FAILED")

PASS  input="'Hello, World!'"                         got=['hello', 'world']
FAIL  input="'PyTorch 2.0 docs'"                      got=['pytorch', '2', '0', 'docs']
PASS  input='"don\'t stop"'                           got=["don't", 'stop']
PASS  input="'multi-head attention'"                  got=['multi-head', 'attention']
PASS  input="'   extra   whitespace   '"              got=['extra', 'whitespace']
PASS  input="''"                                      got=[]

SOME TESTS FAILED


## 2. Database — load records

In [4]:
records = load_records(config.DB_PATH)

print(f'Total records: {len(records)}')
print()

from collections import Counter
label_counts = Counter(label for _, _, label in records)
for label, count in sorted(label_counts.items()):
    print(f'  {label:12s}: {count}')

print()
print('Sample records:')
for url, html_path, label in records[:5]:
    import os
    exists = os.path.exists(html_path)
    print(f'  [{label}] {url}')
    print(f'         html exists={exists}, path={html_path}')

Total records: 20

  productive  : 14
  skip        : 1
  waste       : 5

Sample records:
  [waste] pokemon.com/us
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/04a156a4652be738cdb86bda48b2782d07243738f38e54b61b3fd63d7787e6b2/page.html
  [waste] minecraft.net/en-us
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/fe82e14a01d19579f7d2e1292749c57a417948eb9db1ed9d7ce4c605592bd59e/page.html
  [waste] reddit.com/r/playboicarti/hot/
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/f3062ed76a5d650ff3f49b4cc5b2bf7cd4f3f31a861a8bda582d00eb316f71d2/page.html
  [waste] reddit.com/r/ChainsawMan/
         html exists=True, path=/Users/umair/Library/Application Support/mlops/captures/b68022bc82a9ec114393f681214a2bf46e83365a45702a83610be5beb03fe525/page.html
  [productive] docs.pytorch.org/docs/stable/index.html
         html exists=True, path=/Users/umair/Library/Application Suppo

## 3. HTML Parser — `(token, tag)` pairs

In [5]:
from collections import defaultdict

print(f'Parsing all {len(records)} pages with M={config.M}...\n')

for url, html_path, label in records:
    pairs = _parse_html(html_path, config.M)
    tag_counts = defaultdict(int)
    for _, tag in pairs:
        tag_counts[tag] += 1
    tags_str = ', '.join(f'{t}:{n}' for t, n in sorted(tag_counts.items()))
    print(f'[{label:10s}] {url}')
    print(f'             tokens={len(pairs):4d}  tags: {tags_str}')

Parsing all 20 pages with M=512...

[waste     ] pokemon.com/us
             tokens= 512  tags: em:25, h2:8, h3:102, h5:17, meta_desc:53, p:151, span:149, title:7
[waste     ] minecraft.net/en-us
             tokens= 512  tags: h2:3, h3:15, meta_desc:79, p:293, span:115, title:7
[waste     ] reddit.com/r/playboicarti/hot/
             tokens= 435  tags: h1:2, h2:18, p:321, span:92, title:2
[waste     ] reddit.com/r/ChainsawMan/
             tokens= 512  tags: h1:2, h2:50, p:315, span:143, title:2
[productive] docs.pytorch.org/docs/stable/index.html
             tokens= 512  tags: h1:2, h2:6, p:242, span:237, strong:7, title:18
[productive] chatgpt.com/
             tokens=  51  tags: meta_desc:34, span:16, title:1
[skip      ] youtube.com/
             tokens= 512  tags: meta_desc:31, span:480, title:1
[productive] developer.apple.com/documentation/swift/
             tokens= 512  tags: h2:1, h3:2, meta_desc:25, p:35, span:444, title:5
[productive] developer.apple.com/documentation/swi

In [6]:
# Inspect first 20 (token, tag) pairs from one page
url, html_path, label = records[4]  # change index to inspect a different page
pairs = _parse_html(html_path, config.M)

print(f'Page: {url}  [{label}]')
print(f'Total pairs: {len(pairs)}')
print()
print('First 20 (token, tag) pairs:')
for tok, tag in pairs[:20]:
    print(f'  {tag:12s}  "{tok}"')

Page: docs.pytorch.org/docs/stable/index.html  [productive]
Total pairs: 512

First 20 (token, tag) pairs:
  title         "pytorch"
  title         "documentation"
  title         "pytorch"
  title         "2"
  title         "10"
  title         "documentation"
  span          "opens"
  span          "in"
  span          "a"
  span          "new"
  span          "window"
  span          "opens"
  span          "an"
  span          "external"
  span          "website"
  span          "opens"
  span          "an"
  span          "external"
  span          "website"
  span          "in"


## 4. Vocab + GloVe — `build_corpus_vocab`

In [7]:
token2idx, word_matrix, struct_matrix = build_corpus_vocab(
    records, config.GLOVE_PATH, config.N, config.M
)

print(f'Vocab size:           {len(token2idx)}')
print(f'word_matrix shape:    {word_matrix.shape}   (expected [V, k])')
print(f'struct_matrix shape:  {struct_matrix.shape}  (expected [{config.NUM_TAGS+1}, {config.N}])')
print()

# PAD row must be all zeros
assert word_matrix[0].sum() == 0,   'FAIL: PAD word row is not zero'
assert struct_matrix[0].sum() == 0, 'FAIL: PAD struct row is not zero'
print('PAD rows (index 0): all zeros — OK')

# UNK row must be non-zero (mean of known GloVe vecs)
assert word_matrix[1].sum() != 0, 'FAIL: UNK word row is zero (no GloVe hits?)'
print('UNK row  (index 1): non-zero mean — OK')

Parsing HTML corpus ...


100%|██████████| 20/20 [00:00<00:00, 77.95page/s]


Loading GloVe from /Users/umair/mlops/models/semantic_structure/GloVe/dolma_300_2024_1.2M.100_combined.txt ...


1200000 lines [00:10, 113666.75 lines/s]


GloVe dimension: 300
Vocab: 2469 tokens | GloVe coverage: 2174/2467 (88.1%)
Vocab size:           2469
word_matrix shape:    (2469, 300)   (expected [V, k])
struct_matrix shape:  (13, 16)  (expected [13, 16])

PAD rows (index 0): all zeros — OK
UNK row  (index 1): non-zero mean — OK


In [8]:
# Check a few known words
probe_words = ['pytorch', 'documentation', 'pokemon', 'reddit', 'attention', 'zzzzunknownword']
print(f'{"word":20s}  {"in vocab":10s}  {"GloVe hit":10s}  vec[:4]')
print('-' * 70)
for word in probe_words:
    idx = token2idx.get(word)
    in_vocab = idx is not None
    if idx is not None:
        vec = word_matrix[idx]
        glove_hit = vec.sum() != 0
        preview = str(vec[:4].round(3))
    else:
        glove_hit = False
        preview = '(not in vocab)'
    print(f'{word:20s}  {str(in_vocab):10s}  {str(glove_hit):10s}  {preview}')

word                  in vocab    GloVe hit   vec[:4]
----------------------------------------------------------------------
pytorch               True        True        [ 0.724  0.258  1.298 -0.178]
documentation         True        True        [-0.15  -0.07  -0.276 -0.19 ]
pokemon               True        True        [-0.031 -0.142  0.345  0.166]
reddit                True        True        [-0.001  0.21  -0.327  0.001]
attention             False       False       (not in vocab)
zzzzunknownword       False       False       (not in vocab)


In [9]:
# Structural embedding matrix — each tag should have a distinct non-zero row
print(f'Structural embeddings [{struct_matrix.shape[0]} rows x {struct_matrix.shape[1]} dims]')
print(f'{"idx":4s}  {"tag":12s}  mean    std     first 4 values')
print('-' * 65)

idx_to_tag = {0: '<PAD>'}
idx_to_tag.update({v: k for k, v in config.TAG_TO_IDX.items()})

for i, row in enumerate(struct_matrix):
    tag = idx_to_tag.get(i, '?')
    print(f'{i:4d}  {tag:12s}  {row.mean():+.4f}  {row.std():.4f}  {row[:4].round(4)}')

Structural embeddings [13 rows x 16 dims]
idx   tag           mean    std     first 4 values
-----------------------------------------------------------------
   0  <PAD>         +0.0000  0.0000  [0. 0. 0. 0.]
   1  title         -0.0061  0.2624  [-0.126   0.0545 -0.106  -0.4963]
   2  meta_desc     -0.0240  0.2394  [-0.0933  0.26   -0.1264 -0.1442]
   3  h1            +0.0202  0.2612  [0.3855 0.0592 0.1379 0.1487]
   4  h2            -0.0386  0.2315  [ 0.4568 -0.5269 -0.259  -0.1521]
   5  h3            +0.0311  0.2471  [ 0.2562  0.719  -0.1093 -0.1665]
   6  h4            -0.0059  0.2043  [ 0.0911 -0.2109  0.0105  0.3162]
   7  h5            -0.1650  0.2872  [-1.472e-01 -2.795e-01  1.000e-04 -5.106e-01]
   8  strong        -0.1055  0.2521  [ 0.3089  0.1457  0.0571 -0.4147]
   9  em            -0.0329  0.2689  [-0.0765  0.0718  0.0848  0.1408]
  10  span          -0.0020  0.2354  [-0.4071 -0.1783  0.0694 -0.0891]
  11  p             -0.0307  0.2900  [-0.1725 -0.1375  0.2907  0.2859]
 

## 5. Feature matrix — `extract_page`

In [10]:
url, html_path, label = records[4]  # change index to inspect a different page
R, omega, mask = extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M)

k = word_matrix.shape[1]
n = struct_matrix.shape[1]

print(f'Page: {url}  [{label}]')
print()
print(f'R tuples:     {len(R)}')
print(f'omega shape:  {omega.shape}  (expected [{config.M}, {k+n}])')
print(f'mask shape:   {mask.shape}')
print(f'Real tokens:  {mask.sum()} / {config.M}')
print(f'Padded rows:  {(~mask).sum()}')
print()

# Shape assertions
assert omega.shape == (config.M, k + n), f'omega shape mismatch: {omega.shape}'
assert mask.shape == (config.M,)
assert mask.dtype == bool
print('Shape assertions passed')

# Padding rows must be all-zero
pad_rows = omega[~mask]
assert (pad_rows == 0).all(), 'FAIL: padding rows are not zero'
print('Padding rows are all-zero — OK')

# Concatenation check: omega[i, :k] == word_vec, omega[i, k:] == struct_vec
for i, (tok, word_vec, struct_vec) in enumerate(R[:5]):
    assert np.array_equal(omega[i, :k], word_vec),   f'word_vec mismatch at row {i}'
    assert np.array_equal(omega[i, k:], struct_vec), f'struct_vec mismatch at row {i}'
print('Ω_i = [E_i ; S_j] concatenation check — OK')

Page: docs.pytorch.org/docs/stable/index.html  [productive]

R tuples:     512
omega shape:  (512, 316)  (expected [512, 316])
mask shape:   (512,)
Real tokens:  512 / 512
Padded rows:  0

Shape assertions passed
Padding rows are all-zero — OK
Ω_i = [E_i ; S_j] concatenation check — OK


In [11]:
# Inspect first 10 R tuples
print(f'First 10 R = (t, e, p) tuples from: {url}\n')
print(f'{"token":20s}  {"word_vec[:3]":38s}  struct_vec[:3]')
print('-' * 85)
for tok, word_vec, struct_vec in R[:10]:
    print(f'{tok:20s}  {str(word_vec[:3].round(4)):38s}  {struct_vec[:3].round(4)}')

First 10 R = (t, e, p) tuples from: docs.pytorch.org/docs/stable/index.html

token                 word_vec[:3]                            struct_vec[:3]
-------------------------------------------------------------------------------------
pytorch               [0.724  0.2582 1.298 ]                  [-0.126   0.0545 -0.106 ]
documentation         [-0.1503 -0.0704 -0.2759]               [-0.126   0.0545 -0.106 ]
pytorch               [0.724  0.2582 1.298 ]                  [-0.126   0.0545 -0.106 ]
2                     [-0.2078 -0.2103  0.2284]               [-0.126   0.0545 -0.106 ]
10                    [-0.3736 -0.2026  0.1754]               [-0.126   0.0545 -0.106 ]
documentation         [-0.1503 -0.0704 -0.2759]               [-0.126   0.0545 -0.106 ]
opens                 [-0.3437 -0.0667  0.0892]               [-0.4071 -0.1783  0.0694]
in                    [-0.2735 -0.0274 -0.2061]               [-0.4071 -0.1783  0.0694]
a                     [-0.1813 -0.1594 -0.0779]         

## 6. Timing — single page extraction

In [ ]:
import timeit

url, html_path, label = records[4]  # change index to time a different page

N_RUNS = 1000
elapsed = timeit.timeit(
    lambda: extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M),
    number=N_RUNS,
)

per_call_ms = (elapsed / N_RUNS) * 1000

print(f'Page:        {url}  [{label}]')
print(f'Runs:        {N_RUNS}')
print(f'Total time:  {elapsed:.3f}s')
print(f'Per call:    {per_call_ms:.3f}ms')

## 7. Full corpus extraction

In [ ]:
# Run extract_page on every record and assert correctness
print(f'Extracting features for all {len(records)} pages...\n')
print(f'{"label":12s}  {"url":45s}  tokens  omega_shape')
print('-' * 90)

for url, html_path, label in records:
    R, omega, mask = extract_page(html_path, token2idx, word_matrix, struct_matrix, config.M)

    assert omega.shape == (config.M, k + n)
    assert mask.dtype == bool
    assert (omega[~mask] == 0).all(), f'Non-zero padding in {url}'

    print(f'[{label:10s}]  {url:45s}  {mask.sum():4d}    {omega.shape}')

print()
print('All pages passed')

Extracting features for all 20 pages...

label         url                                            tokens  omega_shape
------------------------------------------------------------------------------------------
[waste     ]  pokemon.com/us                                  512    (512, 316)
[waste     ]  minecraft.net/en-us                             512    (512, 316)
[waste     ]  reddit.com/r/playboicarti/hot/                  435    (512, 316)
[waste     ]  reddit.com/r/ChainsawMan/                       512    (512, 316)
[productive]  docs.pytorch.org/docs/stable/index.html         512    (512, 316)
[productive]  chatgpt.com/                                     51    (512, 316)
[skip      ]  youtube.com/                                    512    (512, 316)
[productive]  developer.apple.com/documentation/swift/        512    (512, 316)
[productive]  developer.apple.com/documentation/swift/swift-standard-library   512    (512, 316)
[productive]  rust-lang.org/learn/                